In [2]:
from pathlib import Path
import json
from transformers import pipeline
from collections import Counter
import random

/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Paths and Data

In [3]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

In [4]:
# Read JSONL file
with open(romance_data_path, 'r') as f:
    romance_stories = [json.loads(line) for line in f]

In [5]:
# Read JSONL file
with open(sci_fi_data_path, 'r') as f:
    sci_fi_stories = [json.loads(line) for line in f]

In [6]:
# Read JSONL file
with open(literary_fiction_data_path, 'r') as f:
    literary_fiction_stories = [json.loads(line) for line in f]

In [7]:
# print mean word count for each genre
def mean_word_count(stories):
    word_counts = [len(story['story'].split()) for story in stories]
    return sum(word_counts) / len(word_counts)

mean_word_count_romance = mean_word_count(romance_stories)
mean_word_count_sci_fi = mean_word_count(sci_fi_stories)
mean_word_count_literary_fiction = mean_word_count(literary_fiction_stories)

print(f"Mean word count for romance stories: {mean_word_count_romance}")
print(f"Mean word count for sci-fi stories: {mean_word_count_sci_fi}")
print(f"Mean word count for literary fiction stories: {mean_word_count_literary_fiction}")

Mean word count for romance stories: 1609.43
Mean word count for sci-fi stories: 1855.72
Mean word count for literary fiction stories: 1768.56


### Genre classifier

In [9]:
# Initialize classifier
classifier = pipeline(
    "zero-shot-classification",
    model="roberta-large-mnli"
)

labels = ["a science fiction story", "a romance story", "a literary fiction story"]

# Sample 100 stories from each genre
sampled_romance = random.sample(romance_stories, 100)
sampled_sci_fi = random.sample(sci_fi_stories, 100)
sampled_lit_fic = random.sample(literary_fiction_stories, 100)

# Function to classify a list of stories
def classify_stories(stories, labels):
    for item in stories:
        result = classifier(
            item["story"],
            candidate_labels=labels,
            multi_label=False
        )
        item["predicted_genre"] = result['labels'][0]
    return stories

# Classify all sampled stories
sampled_romance = classify_stories(sampled_romance, labels)
sampled_sci_fi = classify_stories(sampled_sci_fi, labels)
sampled_lit_fic = classify_stories(sampled_lit_fic, labels)

# Function to calculate and print distribution
def print_distribution(stories, true_genre):
    predictions = [item["predicted_genre"] for item in stories]
    count = Counter(predictions)
    total = len(stories)
    print(f"Distribution for true genre '{true_genre}':")
    for genre, c in count.items():
        print(f"  {genre}: {c / total * 100:.1f}%")
    print()

# Print distributions
print_distribution(sampled_romance, "romance")
print_distribution(sampled_sci_fi, "science fiction")
print_distribution(sampled_lit_fic, "literary fiction")

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


Distribution for true genre 'romance':
  a literary fiction story: 60.0%
  a romance story: 39.0%
  a science fiction story: 1.0%

Distribution for true genre 'science fiction':
  a literary fiction story: 54.0%
  a romance story: 30.0%
  a science fiction story: 16.0%

Distribution for true genre 'literary fiction':
  a romance story: 41.0%
  a literary fiction story: 55.0%
  a science fiction story: 4.0%

